In [1]:
import logging
from datetime import datetime
from typing import Optional, Any
from pydantic import BaseModel, field_validator, ValidationInfo
import uuid

# Configuración básica de logging para simular la DLQ
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("AnomalyDLQ")

class AnomalyService:
    """
    Servicio encargado de registrar datos anómalos en una cola o DB paralela.
    """
    @staticmethod
    def log_anomaly(field: str, value: Any, reason: str, context: Optional[dict] = None):
        """
        Registra la anomalía sin lanzar excepciones.
        En producción, esto podría enviar un mensaje a Kafka, RabbitMQ o insertar en una tabla 'dead_letter'.
        """
        log_entry = {
            "timestamp": datetime.utcnow().isoformat(),
            "field": field,
            "value": value,
            "reason": reason,
            "context": context or {}
        }
        # Simulación de escritura en DB paralela
        logger.warning(f"DLQ_ANOMALY: {log_entry}")
        
        # Aquí iría tu lógica real: await db.anomalies.insert_one(log_entry)

In [2]:
class Usuario(BaseModel):
    nombre: str
    edad: int
    transaction_id: str  # Útil para rastrear el origen de la anomalía

    @field_validator('edad')
    @classmethod
    def validar_edad(cls, v: int, info: ValidationInfo) -> int:
        # 1. Detectar la anomalía
        if v < 0:
            # 2. Registrar en la DLQ sin detener el flujo
            AnomalyService.log_anomaly(
                field="edad",
                value=v,
                reason="Edad negativa detectada",
                context={
                    "usuario_nombre": info.data.get("nombre", "Desconocido"),
                    "transaction_id": info.data.get("transaction_id", "Sin ID")
                }
            )
            
            # 3. Decisión de negocio: ¿Retornamos el valor negativo o lo saneamos?
            # Opción A: Retornar el valor tal cual (permite pasar, pero riesgo downstream)
            # return v 
            
            # Opción B (Recomendada): Sanear el dato para evitar errores posteriores
            # pero marcar que hubo un problema.
            return 0 

        return v

In [3]:
try:
    # Esta transacción NO fallará, pero generará un log en la DLQ
    usuario = Usuario(
        nombre="Juan Pérez",
        edad=-5, 
        transaction_id=str(uuid.uuid4())
    )
    print(f"Usuario creado: {usuario}")
    # Nota: La edad será 0 si usaste la Opción B de saneamiento arriba
except Exception as e:
    print(f"Error crítico: {e}")

Usuario creado: nombre='Juan Pérez' edad=0 transaction_id='83cbf9a1-ad02-4508-8b10-68c9dbab777d'
